# Prepare and transform data in the lakehouse (PySpark)

### Create Delta tables
Run these cells to create Delta tables from raw data using schema-qualified paths (`Tables/dbo/...`).

#### Cell 1 - Spark session configuration
This cell enables two Fabric features that optimize how data is written and read in subsequent cells. V-order optimizes parquet layout for faster reads and better compression. Optimize Write reduces the number of files written and increases individual file size.

In [1]:
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 3, Finished, Available, Finished, False)

#### Cell 2 - Fact - Sale
This cell reads raw parquet data from `Files/wwi-raw-data/full/fact_sale_1y_full`, adds date part columns (`Year`, `Quarter`, and `Month`), and writes `fact_sale` as a Delta table partitioned by `Year` and `Quarter`.

In [2]:
from pyspark.sql.functions import col, year, month, quarter

df = spark.read.format("parquet").load("Files/wwi-raw-data/full/fact_sale_1y_full")
df = df.withColumn("Year", year(col("InvoiceDateKey")))
df = df.withColumn("Quarter", quarter(col("InvoiceDateKey")))
df = df.withColumn("Month", month(col("InvoiceDateKey")))

df.write.mode("overwrite").format("delta").partitionBy("Year", "Quarter").save("Tables/dbo/fact_sale")

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 4, Finished, Available, Finished, False)

#### Cell 3 - Dimensions
This cell reads the five dimension parquet datasets and writes them as Delta tables (`dimension_city`, `dimension_customer`, `dimension_date`, `dimension_employee`, and `dimension_stock_item`) under `Tables/dbo/...`.

In [3]:
def load_full_data_from_source(table_name):
    df = spark.read.format("parquet").load("Files/wwi-raw-data/full/" + table_name)
    if "Photo" in df.columns:
        df = df.drop("Photo")
    df.write.mode("overwrite").format("delta").save("Tables/dbo/" + table_name)

full_tables = [
    "dimension_city",
    "dimension_customer",
    "dimension_date",
    "dimension_employee",
    "dimension_stock_item",
]

for table in full_tables:
    load_full_data_from_source(table)

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 5, Finished, Available, Finished, False)

### Transform data for business aggregates
Transformation cells to create aggregate outputs for reporting in the schema-enabled path.

#### Cell 4 - Load source tables
This cell loads the source Delta tables needed for business aggregate transformations.

In [4]:
df_fact_sale = spark.read.format("delta").load("Tables/dbo/fact_sale")
df_dimension_date = spark.read.format("delta").load("Tables/dbo/dimension_date")
df_dimension_city = spark.read.format("delta").load("Tables/dbo/dimension_city")
df_dimension_employee = spark.read.format("delta").load("Tables/dbo/dimension_employee")

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 6, Finished, Available, Finished, False)

#### Cell 5 - Aggregate sale by date and city
This cell computes monthly sales totals by city and writes the result to `aggregate_sale_by_date_city`.

In [5]:
sale_by_date_city = (
    df_fact_sale.alias("sale")
    .join(df_dimension_date.alias("date"), df_fact_sale.InvoiceDateKey == df_dimension_date.Date, "inner")
    .join(df_dimension_city.alias("city"), df_fact_sale.CityKey == df_dimension_city.CityKey, "inner")
    .select("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "city.City", "city.StateProvince", "city.SalesTerritory", "sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .groupBy("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "city.City", "city.StateProvince", "city.SalesTerritory")
    .sum("sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .withColumnRenamed("sum(TotalExcludingTax)", "SumOfTotalExcludingTax")
    .withColumnRenamed("sum(TaxAmount)", "SumOfTaxAmount")
    .withColumnRenamed("sum(TotalIncludingTax)", "SumOfTotalIncludingTax")
    .withColumnRenamed("sum(Profit)", "SumOfProfit")
    .orderBy("date.Date", "city.StateProvince", "city.City")
)

sale_by_date_city.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/dbo/aggregate_sale_by_date_city")

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 7, Finished, Available, Finished, False)

#### Cell 6 - Aggregate sale by date and employee
This cell computes monthly sales totals by employee and writes the result to `aggregate_sale_by_date_employee`.

In [6]:
sale_by_date_employee = (
    df_fact_sale.alias("sale")
    .join(df_dimension_date.alias("date"), df_fact_sale.InvoiceDateKey == df_dimension_date.Date, "inner")
    .join(df_dimension_employee.alias("employee"), df_fact_sale.SalespersonKey == df_dimension_employee.EmployeeKey, "inner")
    .select("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "employee.PreferredName", "employee.Employee", "sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .groupBy("date.Date", "date.CalendarMonthLabel", "date.Day", "date.ShortMonth", "date.CalendarYear", "employee.PreferredName", "employee.Employee")
    .sum("sale.TotalExcludingTax", "sale.TaxAmount", "sale.TotalIncludingTax", "sale.Profit")
    .withColumnRenamed("sum(TotalExcludingTax)", "SumOfTotalExcludingTax")
    .withColumnRenamed("sum(TaxAmount)", "SumOfTaxAmount")
    .withColumnRenamed("sum(TotalIncludingTax)", "SumOfTotalIncludingTax")
    .withColumnRenamed("sum(Profit)", "SumOfProfit")
    .orderBy("date.Date", "employee.PreferredName", "employee.Employee")
)

sale_by_date_employee.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("Tables/dbo/aggregate_sale_by_date_employee")

StatementMeta(, a5065fdb-e1cb-473b-9996-7ebd9a461df7, 8, Finished, Available, Finished, False)